# Collimator Mesh Definition

Written by Lev Thomson for python 3.9.25

This script generates a mesh for the thruster and collimator using gmsh's python API. 
The mesh can then be imported into LEoPart-x / FEniCSx for use in plasma simulation. 

# Packages / Libraries:

In [1]:
import gmsh # For creating mesh
import numpy as np # For adding magnets
import meshio # For converting to xdmf format for FEniCSx
from collections import defaultdict # For grouping unsorted entities
import json # For mapping names of entities to their tags for import into FEniCSx

# Variables

First, variables are defined for the thruster dimensions, the plasma's Debye length and the dimensions of the mesh. 

In [2]:
import variables

# Functions

The number of magnets used in the collimator varies, and is specified in n_mag. The function add_magnet_pair (defined below) is called repeatedly when generating the mesh. This adds an evenly-spaced set of n_mag*2 magnets around the inner rim of the collimator. 

In [3]:
def add_magnet_pair(
    model: gmsh.model,
    x: float,
    y: float,
    z: float,
    dx: float,
    dy: float,
    dz: float,
    angle: float,
    col_tag:int,
    magnet_volume_list: list,
) -> gmsh.model:
    """
    Adds a pair of magnets at a given point on the collimator-thruster mesh.

    The magnet is a rectangle defined by a point (x\', y\', z\') and the extents along the x-, y- and z-axes, and is rotated by angle.

    The second magnet is added diagonally opposite the first.

    Args:

        model: the model to be updated

        x: x-coordinate of one corner of box

        y: y-coordinate of one corner of box

        z: z-coordinate of one corner of box

        dx: x-coordinate of other corner of box

        dy: y-coordinate of other corner of box

        dz: z-coordinate of other corner of box

        angle: (radians) the rectangle is rotated about the x-axis by this angle

        col_tag: tag of the collimator that the magnets are attached to

        magnet_volume_list: list of all the tags of the magnet volumes. For defining the magnets as a physical group later.



    Returns:

        model: updated gmsh model

        last_tag: the tag of the updated model

        magnet_surface_list: list of magnet surfaces by [(dimension=2, tag)] (for defining field later)
    """

    # Add first magnet

    mag_1_vol = model.occ.addBox(x, -y, -z, dx, dy, dz, -1) #+ 5e-4

    model.occ.rotate([(3, mag_1_vol)], x + dx / 2, 0, 0, 1, 0, 0, angle)

    # Add second magnet opposite the first

    mag_2_vol = model.occ.addBox(x, -y, -z, dx, dy, dz, -1) # + 5e-4

    model.occ.rotate([(3, mag_2_vol)], x + dx / 2, 0, 0, 1, 0, 0, angle + np.pi)

    # Fragment magnets with the collimator - this ensures that the mesh cells in the magnet and the collimator are conformal

    # dt, dtmap = model.occ.cut(
    #     [(3, col_tag)], 
    #     [
    #         (3, mag_1_vol), 
    #         (3, mag_2_vol)
    #         ], 
    #         -1, 
    #         removeObject=True, 
    #         removeTool=False)

    dt, dtmap = model.occ.fragment(
        [(3, col_tag)],
        [(3, mag_1_vol), (3, mag_2_vol)],
    )

    # The magnet and collimators are deleted and recreated when fragmenting. 
    # We then need to find their new tags programatically.

    # dtmap maps the new child volumes to their parents, and is organised like:
    # [
    #  [col_tag's children dimtags],
    #  [mag_1_vol's children dimtags],
    #  [mag_2_vol's children dimtags]
    #  ]

    col_tag = dtmap[0][0][1]
    # mag_1_vol = dtmap[1][0][1]
    # mag_2_vol = dtmap[2][0][1]

    gmsh.model.occ.synchronize()

    # magnet_volume_list.append(mag_1_vol)

    # magnet_volume_list.append(mag_2_vol)
    magnet_volume_list.extend(tag for _, tag in dtmap[1])
    magnet_volume_list.extend(tag for _, tag in dtmap[2])

    model.occ.synchronize()

    return model, col_tag, magnet_volume_list

def add_physical_boundary_surface(
    model: gmsh.model, vol_tag: int | list, phys_tag=int, surf_name=str
) -> int:
    """
    Adds a physical group containing the boundary of the volume vol_tag.

    The resulting physical group has the tag phys_tag, and the name surf_name.

    Args:

        - model: gmsh model to add the physical group to.

        - vol_tag: (positive integer) the tag of the volume whose surface will be added.

        - phys_tag: (positive integer) the tag of the resulting physical group. -1 to let gmsh assign a tag automatically.

        - surf_name: (string) the name of the physical group.



    Returns: the tag of the physical group, phys_tag.
    """
    if type(vol_tag) == int:
        boundary_entities = model.getBoundary([(3, vol_tag)])

        surface_tags = [tag for dim, tag in boundary_entities if dim == 2]
        return model.addPhysicalGroup(2, surface_tags, tag=phys_tag, name=surf_name)
    elif type(vol_tag) == list:
        surface_tags = []
        for vol in vol_tag:
            boundary_entities = model.getBoundary([(3, vol)])
            for dim, tag in boundary_entities:
                if dim == 2:
                    surface_tags.append(tag)
        return model.addPhysicalGroup(2, surface_tags, tag=phys_tag, name=surf_name)

def create_mesh(mesh, cell_type, prune_z=False):
    """
    Modified from code written by J.S. Dokken, December 2021, available here: (https://fenicsproject.discourse.group/t/importing-mesh-gives-error/7282/5)

    This reads in a mesh of type gmsh.mesh, and returns a mesh consisting of only the cells of type cell_type.

    Args:

        - mesh: gmsh.mesh containing mixed cell types

        - cell_type: either "tetra" or "triangle" - the cell type to be included in the output mesh.



    Returns:

        - mesh containing only cells of cell_type.
    """
    cells = mesh.get_cells_type(cell_type)

    cell_data = mesh.get_cell_data(
        "gmsh:physical", cell_type
    )  # For encoding facet tags (required to preserve physical groups)

    points = mesh.points[:, :2] if prune_z else mesh.points

    out_mesh = meshio.Mesh(
        points=points,
        cells={cell_type: cells},
        cell_data={"name_to_read": [cell_data.astype(np.int32)]},
    )

    return out_mesh

# Defining a Model Using GMSH

GMSH stores data as a model, which is an object that contains topological entities. 

These entities are:
Order 0 entities: Points
Order 1 entities: Curves, which are defined by multiple points
Order 2 entities: Surfaces, which are defined by multiple curves
Order 3 entities: Volumes, which are defined by multiple surfaces

Each of these entities is identified by a dimtag, which is a tuple with values (dimension, tag).
The dimension is the order of the entity, and the tag is a positive integer that is unique to that entity within that order. (e.g., [(3,1)] is not the same as [(2,1)]. The first is a volume but the second is a surface.) Tags can be defined when the entity is created, or can be generated automatically by GMSH. 

Various operations are available that allow a user to create a complex shape from simpler entities. These include cut and intersect to remove one entity from another, fuse to join entities, and fragment and getBoundary to divide an order-x entity into its constituent parts of order x-1.  

In [4]:
# ---- Set parameters ----


gmsh.initialize()

gmsh.model.add("collimator")  # Add model with name collimator


# ---- Define geometry ----

vol_coll = gmsh.model.occ.addCylinder(
    variables.l_a, 0, 0, variables.l_coll, 0, 0, variables.r_coll, -1
)  # Collimator cylinder, tag = 1

vol_a = gmsh.model.occ.addCylinder(
    0, 0, 0, variables.l_a, 0, 0, variables.r_a, -1
)  # Anode cylinder, tag = 2

# Then remove the cathode from the anode cylinder

vol_c = gmsh.model.occ.addCylinder(0, 0, 0, variables.l_c, 0, 0, variables.r_c, -1)

vol_trig = gmsh.model.occ.addCylinder(0, 0, 0, variables.l_trig, 0, 0, variables.r_trig, -1)

vol_fragment, vol_fragment_map = gmsh.model.occ.fragment(
    [(3, vol_coll)], 
    [
        (3, vol_a), 
        (3, vol_c),
        (3, vol_trig)
     ], 
     -1)

vol_coll = vol_fragment_map[0][0][1]
vol_a = vol_fragment_map[1][0][1]
vol_c = vol_fragment_map[2][0][1]
vol_trig = vol_fragment_map[3][0][1]
vol_c_trig = vol_fragment_map[3][1][1] 
# The trigger sits within the cathode 
# - this is the section of the trigger that overlaps with the cathode.

gmsh.model.occ.synchronize()


# Add magnets:

if variables.n_mag == 0:
    pass

elif variables.n_mag > 0:
    col_tag = vol_coll  # Tag of most-recently created entity

    angle_array = np.linspace(0, np.pi, variables.n_mag + 1)[
        0:variables.n_mag
    ]  # Generate array of equally spaced magnet pairs around spherical cathode surface

    magnet_volume_list = []

    for loop_idx, angle in enumerate(angle_array):
        gmsh.model, col_tag, magnet_volume_list = add_magnet_pair(
            gmsh.model,
            variables.l_a,
            variables.r_coll,
            variables.w_mag / 2,
            variables.l_mag,
            variables.h_mag,
            variables.w_mag,
            angle,
            col_tag,
            magnet_volume_list,
        )

    vol_coll = col_tag

gmsh.model.occ.synchronize()

magnet_surface_list = []

magnet_names = []
for magtag in magnet_volume_list:
    magnet_names.append((3, magtag))

magnet_surface_list.append(gmsh.model.getBoundary(magnet_names, combined=False, oriented=False))

flat_magnet_surface_tags = [tag for surf in magnet_surface_list for (dim, tag) in surf]

# Defining Physical Groups

GMSH allows users to group entities into "physical groups". This is required to export the .msh file to an .xdmf format for use in FEniCS, but it also makes it possible to define properties for these physical groups. (e.g., all magnets can have the same magnetic properties, which can be different from the collimator wall.)

In [5]:
# ---- Define physical groups -----

print(f"magnet volume: {magnet_volume_list}")


phys_col = gmsh.model.addPhysicalGroup(3, [vol_coll], -1, "collimator")

phys_a = gmsh.model.addPhysicalGroup(3, [vol_a], -1, "anode")

phys_c = gmsh.model.addPhysicalGroup(3, [vol_c], -1, "cathode")

phys_trig = gmsh.model.addPhysicalGroup(3, [vol_trig, vol_c_trig], -1, "trigger")

phys_mag = gmsh.model.addPhysicalGroup(3, magnet_volume_list, -1, "magnets")


phys_surf_col = add_physical_boundary_surface(
    gmsh.model, vol_coll, 54, "collimator_surface"
)

phys_surf_a = add_physical_boundary_surface(gmsh.model, vol_a, -1, "anode_surface")

phys_surf_c = add_physical_boundary_surface(gmsh.model, vol_c, -1, "cathode_surface")

phys_surf_trig = add_physical_boundary_surface(
    gmsh.model, [vol_trig, vol_c_trig], -1, "trigger_surface"
)

phys_surf_mag = gmsh.model.addPhysicalGroup(
    2, flat_magnet_surface_tags, -1, "magnets_surface"
)



magnet volume: [9, 8, 10, 11, 7, 12, 16, 15, 17, 18, 14, 19, 23, 22, 24, 25, 21, 26]


If there are ungrouped entities in the model, this can introduce errors when reading in the file in DOLFINx. Assigning all ungrouped entities to a physical group (sorted by dimension) resolves this. 

In [6]:
entities = gmsh.model.get_entities(-1)

print("Entities")

for e in entities:
    print(f"dim:{e[0]}, tag:{e[1]}")


# Find entities not already part of a physical group and add them to a physical group by dimension

ungrouped_entities = entities

misctags = defaultdict(list)


physicalgroups = gmsh.model.get_physical_groups(-1)

print("Physical groups")

for p in physicalgroups:
    print(f"{gmsh.model.get_physical_name(p[0], p[1])}, dim: {p[0]}, tag = {p[1]}")

    grouped_entities = gmsh.model.get_entities_for_physical_group(p[0], p[1])

    print(f"entities in {gmsh.model.get_physical_name(p[0], p[1])}: {grouped_entities}")

    for g in grouped_entities:
        dt = (p[0], int(g))

        if dt in ungrouped_entities:
            ungrouped_entities.remove(dt)


# Sort ungrouped entities by dimension

for d, e in ungrouped_entities:
    misctags[d].append(e)


phys_misc = defaultdict(int)

phys_names = ["points", "lines", "surfaces", "volumes"]

for ii in range(0, 3):
    if ii in misctags:
        phys_misc[ii] = gmsh.model.addPhysicalGroup(
            ii, misctags[ii], tag=-1, name=phys_names[ii]
        )

        print(
            f"Physical group created for ungrouped {phys_names[ii]}, contains these tags: {misctags[ii]}"
        )


    for t in misctags[ii]:
        dt = (ii, t)

        if dt in ungrouped_entities:
            ungrouped_entities.remove(dt)


# Check again for ungrouped entities: (expect this to print an empty list)

print(f"Ungrouped entities after grouping: {ungrouped_entities}")


gmsh.model.occ.synchronize()

Entities
dim:0, tag:3
dim:0, tag:4
dim:0, tag:5
dim:0, tag:6
dim:0, tag:7
dim:0, tag:8
dim:0, tag:9
dim:0, tag:12
dim:0, tag:13
dim:0, tag:14
dim:0, tag:15
dim:0, tag:16
dim:0, tag:17
dim:0, tag:18
dim:0, tag:19
dim:0, tag:20
dim:0, tag:21
dim:0, tag:22
dim:0, tag:23
dim:0, tag:24
dim:0, tag:25
dim:0, tag:26
dim:0, tag:27
dim:0, tag:28
dim:0, tag:29
dim:0, tag:30
dim:0, tag:31
dim:0, tag:32
dim:0, tag:33
dim:0, tag:34
dim:0, tag:35
dim:0, tag:36
dim:0, tag:37
dim:0, tag:38
dim:0, tag:39
dim:0, tag:42
dim:0, tag:43
dim:0, tag:44
dim:0, tag:45
dim:0, tag:46
dim:0, tag:47
dim:0, tag:48
dim:0, tag:49
dim:0, tag:50
dim:0, tag:51
dim:0, tag:52
dim:0, tag:53
dim:0, tag:54
dim:0, tag:55
dim:0, tag:56
dim:0, tag:57
dim:0, tag:58
dim:0, tag:59
dim:0, tag:60
dim:0, tag:61
dim:0, tag:62
dim:0, tag:63
dim:0, tag:64
dim:0, tag:65
dim:0, tag:66
dim:0, tag:67
dim:0, tag:68
dim:0, tag:69
dim:0, tag:70
dim:0, tag:71
dim:0, tag:72
dim:0, tag:73
dim:0, tag:74
dim:0, tag:75
dim:0, tag:76
dim:0, tag:77
dim:

# Defining Fields

After the geometry of the thruster and collimator has been defined, then a field is applied to the model. This allows users to specify the mesh size programatically. For the plasma simulation, the mesh should be smallest where the magnetic field is strong. For this reason, a threshold field is applied to the mesh. This specifies a minimum mesh size, which occurs within distance DistMin to the magnet surfaces. The mesh size then increases up to a maximum size, which occurs at a distance DistMax from the magnet surfaces. 

Distmax is defined as the Debye length. This is the distance over which an ion's magnetic field has an effect, and is the largest size that a mesh cell should be in the plasma simulation. 

The mesh is then generated by GMSH. This can be a computationally intensive process, so it is done in parallel where possible. The resulting mesh is then saved in a .msh file for importing to FEniCSx. 

If the code is run on a computer that supports graphics, then it is displayed in the GMSH gui automatically. 

In [7]:
# ---- Define mesh field ----

print(f"magnet surfaces: {magnet_surface_list}")


# Get magnet boundary

magnet_array = [] #np.zeros(variables.n_mag * 2 * 6)

for mag_idx, magnet in enumerate(magnet_surface_list):
    for surf_idx, surface in enumerate(magnet):
        magnet_array.append(surface[1]) #[6 * mag_idx + surf_idx] = surface[1]


distance_field = gmsh.model.mesh.field.add("Distance", 1)

gmsh.model.mesh.field.setNumbers(1, "FacesList", magnet_array)

gmsh.model.mesh.field.setNumber(1, "Sampling", 1000)


threshold_field = gmsh.model.mesh.field.add("Threshold", 2)

gmsh.model.mesh.field.setNumber(
    2, "InField", distance_field
)  # Create a mesh field that defines mesh size based on proximity to magnets

gmsh.model.mesh.field.setNumber(2, "SizeMin", variables.size_min)

gmsh.model.mesh.field.setNumber(2, "SizeMax", variables.size_max)

gmsh.model.mesh.field.setNumber(2, "DistMin", 0)

gmsh.model.mesh.field.setNumber(2, "DistMax", 0.05)  # 50mm


gmsh.model.mesh.field.setAsBackgroundMesh(2)


gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)

gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)

gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)


gmsh.model.occ.synchronize()

magnet surfaces: [[(2, 35), (2, 36), (2, 33), (2, 37), (2, 38), (2, 31), (2, 32), (2, 33), (2, 34), (2, 21), (2, 26), (2, 25), (2, 39), (2, 40), (2, 41), (2, 42), (2, 31), (2, 43), (2, 44), (2, 27), (2, 45), (2, 46), (2, 27), (2, 19), (2, 28), (2, 29), (2, 30), (2, 22), (2, 23), (2, 47), (2, 48), (2, 49), (2, 50), (2, 29), (2, 72), (2, 73), (2, 69), (2, 74), (2, 75), (2, 68), (2, 69), (2, 70), (2, 59), (2, 58), (2, 57), (2, 71), (2, 76), (2, 77), (2, 78), (2, 79), (2, 70), (2, 80), (2, 81), (2, 64), (2, 82), (2, 83), (2, 64), (2, 54), (2, 65), (2, 66), (2, 67), (2, 61), (2, 62), (2, 84), (2, 85), (2, 86), (2, 87), (2, 66), (2, 111), (2, 112), (2, 108), (2, 113), (2, 114), (2, 107), (2, 108), (2, 109), (2, 96), (2, 95), (2, 94), (2, 110), (2, 115), (2, 116), (2, 117), (2, 118), (2, 109), (2, 119), (2, 120), (2, 105), (2, 121), (2, 122), (2, 103), (2, 104), (2, 105), (2, 106), (2, 92), (2, 101), (2, 100), (2, 123), (2, 124), (2, 125), (2, 126), (2, 103)]]


# Map Entity Names to Tags

The entities (points, curves, surfaces and volumes) in the gmsh model are defined by their dimtags, which are tuples of unique positive integers. 
The tags are also saved as named variables here, to make the code more readable. 

The entity names are not exported in the MSH or XDMF files, so they are saved here for import into FEniCSx.

In [8]:
# Create a dictionary of entities and tags

physical_groups_dict = defaultdict()

physicalgroups = gmsh.model.get_physical_groups(-1)

for p in physicalgroups:
    name = gmsh.model.get_physical_name(p[0], p[1])
    dim = p[0]
    tag = p[1]
    grouped_entities = gmsh.model.get_entities_for_physical_group(dim, tag)
    entities = [int(e) for e in grouped_entities]

    physical_groups_dict[name] = {
        "dim": dim,
        "tag": tag,
        "entities": entities # list of entity tags
    }

# Export data to a JSON file to read in later
with open("physicalgroups.json", "w") as file:
    json.dump(physical_groups_dict, file, indent=4)



# Generate Mesh

In [9]:
# ---- Parallelize ----


gmsh.option.setNumber(
    "General.NumThreads", 5
)  # Use multithreading to generate mesh faster

gmsh.option.setNumber(
    "Mesh.Algorithm3D", 4
)  # Frontal-Delaunay (default. If this causes issues consider changing to 10 - HXT which is a version of Delaunay that is optimised for parallel meshing).


# ---- Generate Mesh ----

gmsh.option.setNumber("Geometry.Tolerance", 1e-8)

# Remove duplicates (these can cause mesh issues)
gmsh.model.mesh.remove_duplicate_nodes()
gmsh.model.mesh.remove_duplicate_elements()

gmsh.model.mesh.generate()  # (3) # Generate in 3d

gmsh.write("meshes/"+variables.meshname)  # Save mesh as meshname


# ---- Output to GUI ----


if variables.run_gui:  # Off by default for remote server.
    gmsh.fltk.run()


gmsh.finalize()


print("Done")

Info    : Removing duplicate mesh nodes...
Info    : Found 0 duplicate nodes 
Info    : No duplicate nodes found
Info    : Removing duplicate mesh elements...
Info    : Done removing duplicate mesh elements
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 4 (Circle)
Info    : [  0%] Meshing curve 5 (Line)
Info    : [  0%] Meshing curve 6 (Circle)
Info    : [  0%] Meshing curve 7 (Circle)
Info    : [  0%] Meshing curve 8 (Circle)
Info    : [  0%] Meshing curve 9 (Line)
Info    : [  0%] Meshing curve 10 (Circle)
Info    : [  0%] Meshing curve 11 (Line)
Info    : [  0%] Meshing curve 12 (Circle)
Info    : [  0%] Meshing curve 13 (Circle)
Info    : [  0%] Meshing curve 14 (Line)
Info    : [  0%] Meshing curve 17 (Line)
Info    : [  0%] Meshing curve 20 (Line)
Info    : [  0%] Meshing curve 22 (Line)
Info    : [  0%] Meshing curve 23 (Line)
Info    : [  0%] Meshing curve 24 (Line)
Info    : [  0%] Meshing curve 26 (Line)
Info    : [  0%] Meshing curve 27 (Line)
Info    : [  0%] Meshin

Info    : Optimization starts (volume = 0.000622257) with worst = 5.1494e-07 / average = 0.654756:
Info    : 0.00 < quality < 0.10 :       541 elements
Info    : 0.10 < quality < 0.20 :      2729 elements
Info    : 0.20 < quality < 0.30 :      9362 elements
Info    : 0.30 < quality < 0.40 :     15263 elements
Info    : 0.40 < quality < 0.50 :     11125 elements
Info    : 0.50 < quality < 0.60 :     22356 elements
Info    : 0.60 < quality < 0.70 :     66197 elements
Info    : 0.70 < quality < 0.80 :     85040 elements
Info    : 0.80 < quality < 0.90 :     28701 elements
Info    : 0.90 < quality < 1.00 :     11117 elements
Info    : 8538 edge swaps, 2402 node relocations (volume = 0.000622257): worst = 0.0280727 / average = 0.670112 (Wall 0.231082s, CPU 0.231327s)
Info    : 10252 edge swaps, 3079 node relocations (volume = 0.000622257): worst = 0.129743 / average = 0.67288 (Wall 0.319222s, CPU 0.319562s)
Info    : 10560 edge swaps, 3284 node relocations (volume = 0.000622257): worst = 0.

Info    : Done writing 'meshes/collimator.msh'
Done


# Convert to XDMF format

GMSH creates .msh files, which are optimised for use with GMSH only, and are not able to be parallelised. To use the mesh in FEniCSx, the file must be converted to an XDMF and H5 file format, with the volume and surface entities separated into tetra and facet files. This can be done using meshio. 

In [10]:
# This step reads in the msh file so that the cell can be run without the use of gmsh.

in_mesh = meshio.read("meshes/"+variables.meshname)

collimator_tetra = create_mesh(in_mesh, "tetra")
collimator_triangle = create_mesh(in_mesh, "triangle")


# Check that the nodes are sorted correctly.
# All nodes should be unique, and there should be no triangles that are not in the tetra grid.
# Unexpected results here indicate an issue with geometry (probably overlapping meshes). 

print("Number of nodes in grid:", collimator_tetra.points.shape)
tetra_nodes = np.unique(collimator_tetra.cells_dict["tetra"].flatten())
print(
    "Number of unique nodes in grid:",
    len(tetra_nodes),
)

triangle_indices = np.unique(collimator_triangle.cells_dict["triangle"].flatten())
print(
    "Number of triangle nodes that are not in tetra grid:",
    np.sum(np.invert(np.isin(triangle_indices, tetra_nodes))),
)

meshio.write("meshes/collimator_tetra.xdmf", collimator_tetra, compression=None)
meshio.write("meshes/collimator_tetra_facets.xdmf", collimator_triangle, compression=None)

# Mixed-cell file (not used in simulation - just for inspection)
!meshio convert meshes/collimator.msh meshes/collimator.xdmf


import meshio
from scipy.spatial import cKDTree

m = meshio.read("meshes/"+variables.meshname)
pts = m.points
tree = cKDTree(pts)
dists, _ = tree.query(pts, k=2)
print("duplicates already in raw .msh:", np.sum(dists[:, 1] < 1e-9))


Number of nodes in grid: (58298, 3)
Number of unique nodes in grid: 58298
Number of triangle nodes that are not in tetra grid: 0


duplicates already in raw .msh: 0


The mesh is complete. From here, the collimator.msh file can be inspected using the GMSH gui (available at https://www.gmsh.info). 
The mesh is loaded into FEniCSx in the file boundary_conditions.ipynb. 